In [1]:
%pip install -qU langchain-pinecone pinecone-notebooks
%pip install --upgrade --quiet langchain-text-splitters tiktoken
%pip install langchain-openai
%pip install datasets

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or getpass("Enter your OpenAI API key: ")

In [3]:
#retrieve dataframe
import pandas as pd
dataframe = pd.read_pickle("dataframe.pkl")

print(dataframe["text"])


0        so planets become more interesting moons
1             become places to go and revisit but
2           there was a whole other goal and that
3             was the search for intelligent life
4           still is in the universe oh man it is
                           ...                   
77209     All right. This has been Star Talk, the
77210                             Einstein Crumbs
77211       edition. Neil deGrasse Tyson here. As
77212       always, I bid you to keep looking up.
77213                                     [Music]
Name: text, Length: 77214, dtype: object


In [ ]:
# create a new dataframe only with text
docs = dataframe["text"].tolist()
sources = dataframe["source_file"].tolist()

# optionally chunk large documents or sentences
from langchain.text_splitter import CharacterTextSplitter
text_splitter = CharacterTextSplitter(chunk_size=512, chunk_overlap=50)

splits = text_splitter.create_documents(docs)

# Attach metadata separately
for i, doc in enumerate(splits):
    doc.metadata = {"source_file": sources[i]}  # minimal metadata


In [ ]:
# 1. Setup
import pandas as pd
from langchain.embeddings import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore  # ✅ NEW correct import
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv
import os
from getpass import getpass

# Load environment variables
_ = load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") or getpass("Enter your Pinecone API key: ")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or getpass("Enter your OpenAI API key: ")

# Load dataframe and take a small sample
df = pd.read_pickle("dataframe.pkl").head(100)  # Use only the first 100 rows
print(f"✅ Loaded dataframe with {len(df)} rows")
print(df.head(2))  # preview first 2 rows

# Prepare documents and sources
texts = df["text"].tolist()
sources = df["source_file"].tolist()
print(f"✅ Prepared {len(texts)} texts and sources")

# 2. Split texts and attach metadata
splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=20)
documents = [Document(page_content=t, metadata={"source_file": s}) for t, s in zip(texts, sources)]
print(f"✅ Created {len(documents)} Document objects")

split_docs = splitter.split_documents(documents)
print(f"✅ After splitting, got {len(split_docs)} document chunks")
print(f"Sample chunk content:\n{split_docs[0].page_content}")
print(f"Sample chunk metadata:\n{split_docs[0].metadata}")

# 3. Initialize Pinecone v3
index_name = "youtube-transcripts"
dimension = 1536

pc = Pinecone(api_key=PINECONE_API_KEY)

# Create index if needed
if index_name not in [index["name"] for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"✅ Created index: {index_name}")
else:
    print(f"✅ Index '{index_name}' already exists.")

# Connect to the Pinecone Index
index = pc.Index(index_name)
print(f"✅ Connected to Pinecone index: {index_name}")

# Create OpenAI embedding instance
embedding = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
print("✅ Initialized OpenAI embeddings")

# Use langchain_pinecone to store documents
vectordb = PineconeVectorStore.from_documents(
    documents=split_docs,
    embedding=embedding,
    index_name=index_name,
    batch_size=50  # limit batch size to reduce request payload size
)
print(f"✅ Stored {len(split_docs)} documents in Pinecone")

# 4. Set up retriever
retriever = vectordb.as_retriever(search_kwargs={"k": 5})
print("✅ Retriever initialized")

# 5. Memory & Tools
from langchain.memory import ConversationBufferMemory
from langchain.agents import initialize_agent, AgentType, Tool
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA

llm = ChatOpenAI(temperature=0, model="gpt-4", openai_api_key=OPENAI_API_KEY)
print("✅ Initialized LLM")

# Define RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)
print("✅ RetrievalQA chain created")

# Define function for answering questions with sources
def answer_with_sources(input_text: str):
    print(f"\n📝 Query: {input_text}")
    result = qa_chain({"query": input_text})
    answer = result["result"]
    sources = list(set(doc.metadata.get("source_file", "Unknown") for doc in result["source_documents"]))
    print(f"🗒️ Retrieved {len(sources)} source documents")
    return f"{answer}\n\nSources:\n" + "\n".join(sources)


✅ Loaded dataframe with 100 rows
                                       text  \
0  so planets become more interesting moons   
1       become places to go and revisit but   

                                         source_file  
0  40 - Neil deGrasse Tyson and Bill Nye Catch Up...  
1  40 - Neil deGrasse Tyson and Bill Nye Catch Up...  
✅ Prepared 100 texts and sources
✅ Created 100 Document objects
✅ After splitting, got 100 document chunks
Sample chunk content:
so planets become more interesting moons
Sample chunk metadata:
{'source_file': '40 - Neil deGrasse Tyson and Bill Nye Catch Up.en.txt'}
✅ Index 'youtube-transcripts' already exists.
✅ Connected to Pinecone index: youtube-transcripts
✅ Initialized OpenAI embeddings
✅ Stored 100 documents in Pinecone
✅ Retriever initialized
✅ Initialized LLM
✅ RetrievalQA chain created
✅ Tools defined
✅ Conversation memory initialized
✅ Agent initialized and ready


In [ ]:
from langsmith import traceable
import os
!pip install -U langsmith openai

# Set LangSmith env vars
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "LANGCHAIN_API_KEY"
os.environ["LANGCHAIN_PROJECT"] = "youtube-rag"

from langchain.agents import initialize_agent

# Define tool
tools = [
    Tool(
        name="YouTubeTranscriptQA",
        func=answer_with_sources,
        description="Useful for answering questions about YouTube video transcripts. Input should be a fully formed question."
    )
]
print("✅ Tools defined")

# Conversation memory
memory = ConversationBufferMemory(memory_key="chat_history")
print("✅ Conversation memory initialized")

# 6. Initialize Agent with tools and memory
agent = initialize_agent(
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    tools=tools,
    llm=llm,
    verbose=True,
    memory=memory,
    max_iterations=3
)
print("✅ Agent initialized and ready")


@traceable(name="YouTube RAG Trace")
def run_agent():
    agent.run("Your question here")

result = run_agent()



✅ Tools defined
✅ Conversation memory initialized
✅ Agent initialized and ready


> Entering new AgentExecutor chain...


/var/folders/hq/l56ghxv518j9wg6pgqkbbvd80000gn/T/ipykernel_17544/2599423908.py:40: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  agent.run("Your question here")


Thought: Do I need to use a tool? No
AI: I'm sorry, but I need a specific question or topic to provide a response. Could you please provide more details?

> Finished chain.


TypeError: 'NoneType' object is not subscriptable

In [28]:
# Prompt templates for multi-query RAG system

prompt_templates = {
    "summary": "Please provide a concise summary of the following topic: '{}'",
    "source": "Please provide the video source for the following topic: '{}'",
    "explanation": "Explain in detail: '{}'",
    "compare": "Compare and contrast these two concepts: '{}' and '{}'",
    "timeline": "Give me a timeline of events related to: '{}'",
    "faq": "What are the most frequently asked questions about '{}', and their answers?",
    "key_points": "List the key points covered in: '{}'",
    "step_by_step": "Provide a step-by-step guide on how to: '{}'",
    "examples": "Give me examples related to '{}'",
    "pros_cons": "What are the pros and cons of '{}'",
    "common_mistakes": "What are the common mistakes people make regarding '{}', and how to avoid them?",
}

def ask_agent(agent, prompt_type, *args):
    """
    Send a prepared prompt to the agent.
    
    Parameters:
        agent: Your initialized LangChain agent instance
        prompt_type (str): key of the prompt_templates dict
        *args: arguments to format the prompt string
    
    Returns:
        response (str): the agent's answer
    """
    if prompt_type not in prompt_templates:
        raise ValueError(f"Prompt type '{prompt_type}' not supported.")
    prompt = prompt_templates[prompt_type].format(*args)
    print(f"Prompt to model:\n{prompt}\n")
    response = agent.run(prompt)
    return response



In [29]:
# Example usage:
result = ask_agent(agent, "summary", "quantum computing")
print(result)

Prompt to model:
Please provide a concise summary of the following topic: 'quantum computing'



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? No
AI: Quantum computing is a type of computation that leverages the principles of quantum mechanics. Unlike classical computers that use bits (0s and 1s) as their smallest unit of information, quantum computers use quantum bits, or "qubits". Qubits can exist in a state of superposition, where they can be both 0 and 1 at the same time. This, along with the ability to leverage quantum entanglement, where qubits can be instantaneously connected regardless of distance, allows quantum computers to process information in a fundamentally different way. This could potentially lead to significant advancements in areas such as cryptography, material science, and complex problem solving. However, the field is still in its early stages and faces many technical challenges.

> Finished chain.
Quantum computing is a type of computati

In [30]:
result = ask_agent(agent, "source", "Bill Nye")
print(result)

Prompt to model:
Please provide the video source for the following topic: 'Bill Nye'



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: YouTubeTranscriptQA
Action Input: What are some videos about 'Bill Nye'?
📝 Query: What are some videos about 'Bill Nye'?


/var/folders/hq/l56ghxv518j9wg6pgqkbbvd80000gn/T/ipykernel_17544/1846901959.py:203: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain({"query": input_text})


🗒️ Retrieved 1 source documents

Observation: I'm sorry, but the provided context does not include information about videos about 'Bill Nye'.

Sources:
40 - Neil deGrasse Tyson and Bill Nye Catch Up.en.txt
Thought:Do I need to use a tool? No
AI: One video source that discusses 'Bill Nye' is "Neil deGrasse Tyson and Bill Nye Catch Up". This video features a conversation between Neil deGrasse Tyson and Bill Nye.

> Finished chain.
One video source that discusses 'Bill Nye' is "Neil deGrasse Tyson and Bill Nye Catch Up". This video features a conversation between Neil deGrasse Tyson and Bill Nye.


In [ ]:
result = ask_agent(agent, "summary", "quantum computing")
